# Wine Quality Neural Network Classifier

**Student Name**: Meng Oudom  
**Lab**: 05

## Goal
Build a neural network classifier to predict wine quality (Medium, Good, Excellent). 
The dataset is divided into Training (60%), Validation (10%), and Testing (30%).
Models are selected based on validation performance, tuning hidden layers and units.

## 1. Imports

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

# Set random seed for reproducibility
RANDOM_SEED = 42

## 2. Data Loading and Inspection

In [8]:
# Load the dataset
df = pd.read_csv('wine_quality.csv')

# 1. How many data?
print(f"Data Shape (Rows, Columns): {df.shape}")

# 2. What the data looks like?
print("\nFirst 5 rows of data:")
display(df.head())

# 3. Data Cleaning Inspection
print("\nMissing Values per column:")
print(df.isnull().sum())

# Drop duplicates if any
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Duplicates dropped. New shape: {df.shape}")

Data Shape (Rows, Columns): (4898, 12)

First 5 rows of data:


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,Good
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,Good
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,Good
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,Good
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,Good



Missing Values per column:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

Duplicate rows found: 937
Duplicates dropped. New shape: (3961, 12)


## 3. Preprocessing

In [9]:
# Separate features and target
X = df.drop('quality', axis=1)
y = df['quality']

# Encode the target labels
# Map: Medium -> 0, Good -> 1, Excellent -> 2 (or let LabelEncoder decide, but manual is safer for order)
quality_mapping = {'Medium': 0, 'Good': 1, 'Excellent': 2}
y_encoded = y.map(quality_mapping)

# Check if mapping worked (handle potential unmapped values)
if y_encoded.isnull().any():
    print("Warning: Some target values were not mapped correctly. Checking unique values:")
    print(y.unique())
    # Fallback with LabelEncoder if manual mapping fails
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print("Classes:", le.classes_)

# Standardize the features (Neural Networks require scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled and target encoded.")

Features scaled and target encoded.


## 4. Data Splitting
We need: 
- Training: 60%
- Validation: 10%
- Testing: 30%

Since `train_test_split` only splits into two, we perform it twice.
1. Split (Train+Val) and Test (70% / 30%)
2. Split Train and Val from the 70% chunk (60% total is 6/7 of 70%, 10% total is 1/7 of 70%)

In [10]:
# First split: Separate out the 30% Test set
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y_encoded, test_size=0.3, random_state=RANDOM_SEED, stratify=y_encoded)

# Second split: Split the remaining 70% into Train (60% total) and Validation (10% total)
# To get 10% from the original (which is 1/7 of the remaining 70%), we use test_size=1/7
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=1/7, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Training set shape:   {X_train.shape} ({X_train.shape[0]/len(df):.1%} of data)")
print(f"Validation set shape: {X_val.shape}   ({X_val.shape[0]/len(df):.1%} of data)")
print(f"Testing set shape:    {X_test.shape}  ({X_test.shape[0]/len(df):.1%} of data)")

Training set shape:   (2376, 11) (60.0% of data)
Validation set shape: (396, 11)   (10.0% of data)
Testing set shape:    (1189, 11)  (30.0% of data)


## 5. Model Selection (Grid Search)
We will test architectures with:
- Hidden Layers: 2, 3, 4
- Units per Layer: 25, 50, 100

In [11]:
hidden_layers_options = [2, 3, 4]
units_options = [25, 50, 100]

results = []
best_val_error = float('inf')
best_model_params = None
best_model = None

print(f"{'Layers':<10} {'Units':<10} {'Val Error':<10}")
print("-"*30)

for layers in hidden_layers_options:
    for units in units_options:
        # Construct the hidden_layer_sizes tuple, e.g., (50, 50, 50) for 3 layers of 50 units
        hidden_layer_sizes = tuple([units] * layers)
        
        # Initialize MLPClassifier
        # Using max_iter=2000 to ensure convergence
        mlp = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, 
                            max_iter=2000, 
                            random_state=RANDOM_SEED)
        
        # Train on Training set
        mlp.fit(X_train, y_train)
        
        # Predict on Validation set
        y_val_pred = mlp.predict(X_val)
        
        # Calculate Error Rate (1 - Accuracy)
        val_error = 1 - accuracy_score(y_val, y_val_pred)
        
        results.append({
            'layers': layers,
            'units': units,
            'val_error': val_error
        })
        
        print(f"{layers:<10} {units:<10} {val_error:.4f}")
        
        # Check if this is the best model so far
        if val_error < best_val_error:
            best_val_error = val_error
            best_model_params = (layers, units)
            best_model = mlp

print("-"*30)
print(f"Best Architecture: {best_model_params[0]} Layers, {best_model_params[1]} Units")
print(f"Best Validation Error: {best_val_error:.4f}")

Layers     Units      Val Error 
------------------------------
2          25         0.4722
2          50         0.4419
2          100        0.4470
3          25         0.4874
3          50         0.4470
3          100        0.4571
4          25         0.4773
4          50         0.4823
4          100        0.4520
------------------------------
Best Architecture: 2 Layers, 50 Units
Best Validation Error: 0.4419


## 6. Final Evaluation
Evaluate the selected best model on the Test set.

In [12]:
# Predict on Test Set
y_test_pred = best_model.predict(X_test)

# Compute Error Rate
test_error = 1 - accuracy_score(y_test, y_test_pred)

print(f"Final Test Set Error Rate: {test_error:.4f}")
print(f"Final Test Set Accuracy:   {1-test_error:.4f}")

Final Test Set Error Rate: 0.4575
Final Test Set Accuracy:   0.5425
